In [ ]:
##Installation librairie 
##pip install langchain langchain-openai faiss-cpu pypdf streamlit

In [ ]:
#!/usr/bin/env python
##"""
##RAG.py
##────────────────────────────
##Minimal Retrieval-Augmented-Generation demo with:

 ## • RecursiveCharacterTextSplitter  (LangChain utility only)
 ## • PDF + TXT ingestion
 ## • FAISS vector search
 ## • OpenAI embeddings + chat
 ## • Chat history inside the loop

##Dependencies
##------------
##pip install openai==1.* faiss-cpu numpy pypdf langchain tqdm
##"""

In [1]:
## Import de bibliothèque 
from dotenv import load_dotenv
import openai
import os
# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("openai_key")

In [2]:
#Import de bibliothèque 
from __future__ import annotations
import os, json, textwrap
from pathlib import Path
import faiss, numpy as np, openai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader               
from tqdm.auto import tqdm              

# ─────────────────────────── Config ──────────────────────────────
DOCS_DIR       = Path("C:\\Users\\j_aka\\Desktop\\Projet IA GENERATIVE\\Data")    
EMBED_MODEL    = "text-embedding-3-small"
CHAT_MODEL     = "gpt-4o-mini"
CHUNK_SIZE     = 1000                     
CHUNK_OVERLAP  = 200
TOP_K          = 10

SYSTEM_PROMPT = (
    "You are a precise, concise tutor. "
    "Answer ONLY from the provided context. "
    "If the answer is missing, say “I don't know.”"
)

openai.api_key = OPENAI_API_KEY
assert openai.api_key, "👉  Please set OPENAI_API_KEY first!"
# ─────────────────────────────────────────────────────────────────

# Importer & diviser mes pdfs
# Fonction pour lire mes fichier pdf page par page  ----------------------------------------------------------
def read_pdf_text(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

# Fonction pour diviser les fichiers pdf page par page  ----------------------------------------------------------
def load_and_split() -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )
    chunks: list[str] = []
    for path in DOCS_DIR.rglob("*"):
        if path.suffix.lower() == ".txt":
            text = path.read_text(encoding="utf-8", errors="ignore")
        elif path.suffix.lower() == ".pdf":
            text = read_pdf_text(path)
        else:
            continue
        chunks.extend(splitter.split_text(text))
    if not chunks:
        raise RuntimeError(f"No .txt or .pdf files found inside {DOCS_DIR}")
    return chunks

c:\Users\j_aka\anaconda3\envs\AI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## Appel de la fonction de chargement des fichiers sources et division en segements 
chunks = load_and_split()
print(f"Loaded {len(chunks)} text chunks from {DOCS_DIR}.")

Loaded 2227 text chunks from C:\Users\j_aka\Desktop\Projet IA GENERATIVE\Data.


In [4]:
# OpenAI vectorisation ------------------------------------------------------
def embed(texts: list[str]) -> list[list[float]]:
    res = openai.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in res.data]

In [5]:
# Chargement ou indexation  --------------------------------------------------
def get_faiss_store(chunks: list[str], idx_path: str = "faiss.index"):
    meta_path = idx_path + ".meta.json"
    if Path(idx_path).exists() and Path(meta_path).exists():
        print("✓  Loading existing vector store …")
        index = faiss.read_index(idx_path)
        chunks = json.loads(Path(meta_path).read_text())
        return index, chunks

    print("⏳  Building vector store …")
    all_vectors = []
    for i in tqdm(range(0, len(chunks), 128), unit="batch"):
        all_vectors.extend(embed(chunks[i : i + 128]))
    mat = np.asarray(all_vectors, dtype=np.float32)

    index = faiss.IndexFlatL2(mat.shape[1])
    index.add(mat)
    faiss.write_index(index, idx_path)
    Path(meta_path).write_text(json.dumps(chunks))
    return index, chunks

In [6]:
get_faiss_store(chunks)

⏳  Building vector store …


100%|██████████| 18/18 [00:21<00:00,  1.21s/batch]


(<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000001C80165E190> >,
 ['Règles Services Système Fréquence  \nVersion applicable au 1 er  Septembre 2022  \n \n \n \n Préambule 2 \n \n1 PREAMBULE ....................................................................................................................................... 6 \n1.1 OBJET ET PERIMETRE DES REGLES ............................................................................................................ 6 \n1.2 CADRE JURIDIQUE ........................................................................................................................................ 6 \n2 DEFINITIONS ...................................................................................................................................... 8 \n3 DISPOSITIONS GENERALES ............................................................................................................ 15',
  '3 DISPOSITIONS

In [7]:
# Récupération du pipeline RAG. --------------------------------------------------------------
def retrieve(query: str, index, chunks, k: int = TOP_K) -> list[str]:
    q_vec = np.asarray(embed([query])[0], dtype=np.float32).reshape(1, -1)
    _, idxs = index.search(q_vec, k)
    return [chunks[i] for i in idxs[0]]

In [8]:
# Construiction du prompot final ---------------------------------------------------------
def build_user_prompt(question: str, ctx_chunks: list[str]) -> str:
    context_block = "\n\n".join(
        f"[Doc {i+1}]\n{chunk}" for i, chunk in enumerate(ctx_chunks)
    )
    return (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {question}\nAnswer:"
        f"answer in don't know if the answer is not in the context"
    )


In [9]:

# Boucle de Chat avec Historique RAG -------------------------------------------------
def chat_loop(index, chunks):
    history: list[dict] = []                 # stores past turns (user & assistant)
    system_msg = {"role": "system", "content": SYSTEM_PROMPT}

    while True:
        try:
            q = input("\n💬  Ask (Ctrl-C to quit): ")
        except KeyboardInterrupt:
            print("\nBye!")
            break

        ctx = retrieve(q, index, chunks)
        user_prompt = build_user_prompt(q, ctx)

        # Combine system prompt, history, and current user input
        messages = [system_msg] + history + [{"role": "user", "content": user_prompt}]

        # Show the retrieved context (for teaching transparency)
        print("\n🔍  Retrieved context:")
        print("─" * 60)
        for i, c in enumerate(ctx, 1):
            print(textwrap.indent(textwrap.fill(c, width=88), f"[Doc {i}] "))
        print("─" * 60)

        # Call the chat model
        response = openai.chat.completions.create(
            model=CHAT_MODEL, messages=messages, temperature=0.2
        )
        answer = response.choices[0].message.content

        print("🤖  Answer:\n")
        print(textwrap.fill(answer, width=88))

        # Update history with the *plain* question and answer (omit long context)
        history.extend([
            {"role": "user", "content": q},
            {"role": "assistant", "content": answer},
        ])

In [10]:
if __name__ == "__main__":
    chunks = load_and_split()
    index, chunks = get_faiss_store(chunks)
    chat_loop(index, chunks)

✓  Loading existing vector store …

🔍  Retrieved context:
────────────────────────────────────────────────────────────
[Doc 1] d’Injection raccordés à son réseau de distribution qui participent au MA au sein d’une
[Doc 1] EDA Injection.  La mise à jour de ce fichier est envoyée tous les Mois Civils (sauf cas
[Doc 1] d’exception décrit ci-après).  6.3.2 Nom du fichier  N° Champ  Format  1 Le type du
[Doc 1] fichier "MA_REFINJ_GRD" (en majuscules).  2 Le mois de validité de ces informations Un
[Doc 1] mois sous la forme "AAAAMM".  3 Le code EIC du GRD de raccordement  du site   Un code
[Doc 1] EIC.  Le GRD de raccordement du site peut  éventuellement être mandant auprès d’un
[Doc 1] autre GRD.  4 La date et l'heure de création du fichier Une horodate sous la forme
[Doc 1] "AAAAMMJJhhmmss".  5 L'extension du fichier ".csv" (en minuscules).   La forme générale
[Doc 1] du nom du fichier est :   MA_REFINJ_GRD_[Mois de validité]_[Code EIC du GRD]_[Horodate
[Doc 1] de création].csv    6.3.3 Li

BadRequestError: Error code: 400 - {'error': {'message': "'$.input' is invalid. Please check the API reference: https://platform.openai.com/docs/api-reference.", 'type': 'invalid_request_error', 'param': None, 'code': None}}